# Comparação de desempenho por dataset (Plotly)

Este notebook compara os resultados:

- **Global**: `m_30_baseline_vs_kdehist_global.csv` (exp_015) vs `mn_30_bcts.csv` (exp_013)
- **PerClass**: `m_30_baseline_vs_kdehist_perclass.csv` (exp_015) vs `mn_30_bcts_perclass.csv` (exp_013)

As visualizações focam em **erro por dataset**.


In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

REPO_ROOT = Path.cwd()

def find_existing(*candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

GLOBAL_BASELINE_PATH = find_existing(
    REPO_ROOT / 'm_30_baseline_vs_kdehist_global.csv',
    REPO_ROOT / 'experiments' / 'exp_015' / 'm_30_baseline_vs_kdehist_global.csv',
)
GLOBAL_BCTS_PATH = find_existing(
    REPO_ROOT / 'mn_30_bcts.csv',
    REPO_ROOT / 'experiments' / 'exp_013' / 'mn_30_bcts.csv',
)

PERCLASS_BASELINE_PATH = find_existing(
    REPO_ROOT / 'm_30_baseline_vs_kdehist_perclass.csv',
    REPO_ROOT / 'experiments' / 'exp_015' / 'm_30_baseline_vs_kdehist_perclass.csv',
)
PERCLASS_BCTS_PATH = find_existing(
    REPO_ROOT / 'mn_30_bcts_perclass.csv',
    REPO_ROOT / 'experiments' / 'exp_013' / 'mn_30_bcts_perclass.csv',
)

print('GLOBAL_BASELINE_PATH:', GLOBAL_BASELINE_PATH)
print('GLOBAL_BCTS_PATH    :', GLOBAL_BCTS_PATH)
print('PERCLASS_BASELINE_PATH:', PERCLASS_BASELINE_PATH)
print('PERCLASS_BCTS_PATH    :', PERCLASS_BCTS_PATH)


In [ ]:
def load_results(path, source_name):
    if path is None:
        raise FileNotFoundError(f'Arquivo não encontrado para {source_name}')

    df = pd.read_csv(path)
    required_cols = {'dataset', 'modelo', 'erro'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'{path} sem colunas obrigatórias: {missing}')

    out = df.copy()
    out['source'] = source_name
    return out


def prepare_pair(df_a, df_b):
    df = pd.concat([df_a, df_b], ignore_index=True)
    # Agregação por dataset+modelo+source
    agg = (
        df.groupby(['dataset', 'modelo', 'source'], as_index=False)
          .agg(erro_mean=('erro', 'mean'), erro_median=('erro', 'median'), n=('erro', 'size'))
    )
    return df, agg


## 1) Comparação Global


In [ ]:
global_baseline = load_results(GLOBAL_BASELINE_PATH, 'exp_015_global')
global_bcts = load_results(GLOBAL_BCTS_PATH, 'exp_013_global')

global_raw, global_agg = prepare_pair(global_baseline, global_bcts)

display(global_agg.head())
print('Datasets (global):', global_agg['dataset'].nunique())
print('Modelos (global):', sorted(global_agg['modelo'].unique()))


In [ ]:
# Boxplot das distribuições de erro por modelo
fig = px.box(
    global_raw,
    x='modelo',
    y='erro',
    color='source',
    points='outliers',
    title='Global: distribuição de erro por modelo (exp_015 vs exp_013)'
)
fig.update_layout(xaxis_title='Modelo', yaxis_title='Erro absoluto')
fig.show()


In [ ]:
# Heatmap de erro médio por dataset x modelo
pivot_global = global_agg.pivot_table(
    index='dataset',
    columns='modelo',
    values='erro_mean',
    aggfunc='mean'
).sort_index()

fig = px.imshow(
    pivot_global,
    aspect='auto',
    color_continuous_scale='Viridis',
    title='Global: erro médio por dataset e modelo'
)
fig.update_layout(xaxis_title='Modelo', yaxis_title='Dataset')
fig.show()


In [ ]:
# Barras agrupadas por dataset (média), separando por source
plot_df = global_agg.copy()

fig = px.bar(
    plot_df,
    x='dataset',
    y='erro_mean',
    color='modelo',
    facet_row='source',
    barmode='group',
    height=900,
    title='Global: erro médio por dataset (exp_015 vs exp_013)'
)
fig.update_xaxes(tickangle=45)
fig.update_layout(yaxis_title='Erro médio')
fig.show()


## 2) Comparação PerClass


In [ ]:
perclass_baseline = load_results(PERCLASS_BASELINE_PATH, 'exp_015_perclass')
perclass_bcts = load_results(PERCLASS_BCTS_PATH, 'exp_013_perclass')

perclass_raw, perclass_agg = prepare_pair(perclass_baseline, perclass_bcts)

display(perclass_agg.head())
print('Datasets (perclass):', perclass_agg['dataset'].nunique())
print('Modelos (perclass):', sorted(perclass_agg['modelo'].unique()))


In [ ]:
fig = px.box(
    perclass_raw,
    x='modelo',
    y='erro',
    color='source',
    points='outliers',
    title='PerClass: distribuição de erro por modelo (exp_015 vs exp_013)'
)
fig.update_layout(xaxis_title='Modelo', yaxis_title='Erro absoluto')
fig.show()


In [ ]:
pivot_perclass = perclass_agg.pivot_table(
    index='dataset',
    columns='modelo',
    values='erro_mean',
    aggfunc='mean'
).sort_index()

fig = px.imshow(
    pivot_perclass,
    aspect='auto',
    color_continuous_scale='Plasma',
    title='PerClass: erro médio por dataset e modelo'
)
fig.update_layout(xaxis_title='Modelo', yaxis_title='Dataset')
fig.show()


In [ ]:
fig = px.bar(
    perclass_agg,
    x='dataset',
    y='erro_mean',
    color='modelo',
    facet_row='source',
    barmode='group',
    height=900,
    title='PerClass: erro médio por dataset (exp_015 vs exp_013)'
)
fig.update_xaxes(tickangle=45)
fig.update_layout(yaxis_title='Erro médio')
fig.show()


## 3) Tabelas resumo (Top datasets)


In [ ]:
def top_datasets(agg, top_k=15):
    rank = (
        agg.groupby(['dataset', 'source'], as_index=False)
           .agg(erro_medio_geral=('erro_mean', 'mean'))
           .sort_values(['source', 'erro_medio_geral'])
    )
    return rank.groupby('source').head(top_k)

print('=== Global: melhores datasets por source ===')
display(top_datasets(global_agg, top_k=15))

print('=== PerClass: melhores datasets por source ===')
display(top_datasets(perclass_agg, top_k=15))
